In [2]:
# Block 1 — setup + read IPUMS XML schema + read raw .dat
import xml.etree.ElementTree as ET
from pyspark.sql import SparkSession, functions as F

#spark = SparkSession.builder.getOrCreate()

spark = SparkSession.builder \
    .config("spark.driver.memory", "2g") \
    .config("spark.executor.memory", "4g") \
    .config('spark.executor.instances', 63) \
    .getOrCreate()

xml_path = "../../evargasnavarro/shared/processed/usa_00001.xml"
dat_path = "../../evargasnavarro/shared/processed/usa_00001.dat"

# Parse IPUMS DDI XML for fixed-width specs
ns = {"ddi": "ddi:codebook:2_5"}
root = ET.parse(xml_path).getroot()

specs = []
for var in root.findall(".//ddi:dataDscr/ddi:var", ns):
    loc = var.find("ddi:location", ns)
    fmt = var.find("ddi:varFormat", ns)
    specs.append({
        "name": var.attrib["name"],
        "start": int(loc.attrib["StartPos"]),
        "width": int(loc.attrib["width"]),
        "dcml": int(var.attrib.get("dcml", "0")),
        "type": fmt.attrib.get("type", "character") if fmt is not None else "character"
    })

raw_df = spark.read.text(dat_path)
raw_df.show(5, truncate=False)  # raw fixed-width lines
print(f"Columns in XML spec: {len(specs)}")

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [3]:
# Block 2 — parse to Spark DataFrame + cast numerics + inspect
df = raw_df.select(*[
    F.substring("value", s["start"], s["width"]).alias(s["name"])
    for s in specs
])

for s in specs:
    if s["type"] == "numeric":
        df = df.withColumn(s["name"], F.col(s["name"]).cast("double"))
        if s["dcml"] > 0:
            df = df.withColumn(s["name"], F.col(s["name"]) / (10 ** s["dcml"]))

# readable checks
print("rows:", df.count())
print("cols:", len(df.columns))
df.select("YEAR", "STATEFIP", "SEX", "AGE", "RACE", "EDUC", "INCTOT").show(10, truncate=False)
df.printSchema()

rows: 67125780
cols: 238
+------+--------+---+----+----+----+---------+
|YEAR  |STATEFIP|SEX|AGE |RACE|EDUC|INCTOT   |
+------+--------+---+----+----+----+---------+
|2001.0|1.0     |1.0|39.0|1.0 |3.0 |6400.0   |
|2001.0|1.0     |1.0|33.0|2.0 |10.0|30000.0  |
|2001.0|1.0     |2.0|40.0|1.0 |6.0 |16500.0  |
|2001.0|1.0     |1.0|10.0|1.0 |1.0 |9999999.0|
|2001.0|1.0     |2.0|21.0|1.0 |3.0 |0.0      |
|2001.0|1.0     |1.0|39.0|1.0 |6.0 |21000.0  |
|2001.0|1.0     |2.0|55.0|1.0 |10.0|12000.0  |
|2001.0|1.0     |1.0|38.0|1.0 |3.0 |20000.0  |
|2001.0|1.0     |2.0|32.0|1.0 |2.0 |0.0      |
|2001.0|1.0     |1.0|10.0|1.0 |1.0 |9999999.0|
+------+--------+---+----+----+----+---------+
only showing top 10 rows

root
 |-- YEAR: double (nullable = true)
 |-- SAMPLE: double (nullable = true)
 |-- SERIAL: double (nullable = true)
 |-- CBSERIAL: double (nullable = true)
 |-- NUMPREC: double (nullable = true)
 |-- SUBSAMP: double (nullable = true)
 |-- HHWT: double (nullable = true)
 |-- EXPWTH: double 

# Data Exploration using Spark

In [4]:
print("Number of Rows:", df.count())
print("Number of Columns:", len(df.columns))

Number of Rows: 67125780
Number of Columns: 238


From the schema printed above, our target columns are "YEAR", "STATEFIP", "SEX", "AGE", "RACE", "EDUC", and "INCTOT". All categorical variables have some numeric coding scheme which correspond with qualitative categories. The coding scheme for these variables can be found in the code box below. These coding schemes and descriptions are found on the [IPUMS website](https://usa.ipums.org/usa-action/variables/group).
- YEAR (Numerical): The year the data was collected. This is for giving a chronological overview of the day.
- STATEFIP (Categorical): The US state the data was collected using the FIPS (Federal Information Processing Standards) coding scheme.
- SEX (Categorical): Whether the person was male or female
- AGE (Numerical): The person's age in years.
- RACE (Categorical): The person's race.
- EDUC (Categorical): The person's educational attainment
- INCTOT (Numerical): The person's pre-tax total personal income (or loss).

In [5]:
statefip = {
    1: "Alabama",
    2: "Alaska",
    4: "Arizona",
    5: "Arkansas",
    6: "California",
    8: "Colorado",
    9: "Connecticut",
    10: "Delaware",
    11: "District of Columbia",
    12: "Florida",
    13: "Georgia",
    15: "Hawaii",
    16: "Idaho",
    17: "Illinois",
    18: "Indiana",
    19: "Iowa",
    20: "Kansas",
    21: "Kentucky",
    22: "Louisiana",
    23: "Maine",
    24: "Maryland",
    25: "Massachusetts",
    26: "Michigan",
    27: "Minnesota",
    28: "Mississippi",
    29: "Missouri",
    30: "Montana",
    31: "Nebraska",
    32: "Nevada",
    33: "New Hampshire",
    34: "New Jersey",
    35: "New Mexico",
    36: "New York",
    37: "North Carolina",
    38: "North Dakota",
    39: "Ohio",
    40: "Oklahoma",
    41: "Oregon",
    42: "Pennsylvania",
    44: "Rhode Island",
    45: "South Carolina",
    46: "South Dakota",
    47: "Tennessee",
    48: "Texas",
    49: "Utah",
    50: "Vermont",
    51: "Virginia",
    53: "Washington",
    54: "West Virginia",
    55: "Wisconsin",
    56: "Wyoming",
    72: "Puerto Rico",
    99: "State not identified"
}
sex = {
    1: "Male", 
    2: "Female", 
    9: "Missing/blank"
}
race = {
    1: "White", 
    2: "Black/African American", 
    3: "American Indian or Alaska Native", 
    4: "Chinese", 
    5: "Japanese", 
    6: "Other Asian or Pacific Islander", 
    7: "Other race, nec", 
    8: "Two major races", 
    9: "Three or more major races"
}
educ = {
    0: "N/A or no schooling", 
    1: "Nursery school to grade 4", 
    2: "Grade 5, 6, 7, or 8", 
    3: "Grade 9", 
    4: "Grade 10", 
    5: "Grade 11", 
    6: "Grade 12", 
    7: "1 year of college", 
    8: "2 years of college", 
    9: "3 years of college", 
    10: "4 years of college", 
    11: "5+ years of college", 
    99: "Missing"
}

In [6]:
target_columns = ["YEAR", "STATEFIP", "SEX", "AGE", "RACE", "EDUC", "INCTOT"]
df = df.select(target_columns)

### Descriptive statistics for Numeric data

In [15]:
described = df.select(["YEAR", "AGE", "INCTOT"]).describe()

In [16]:
described.show(truncate=False)

+-------+------------------+------------------+------------------+
|summary|YEAR              |AGE               |INCTOT            |
+-------+------------------+------------------+------------------+
|count  |67125780          |67125780          |67125780          |
|mean   |2013.8564116796856|40.78647914407848 |1774229.3693289524|
|stddev |6.378003965847706 |23.575137985239095|3777923.4428676735|
|min    |2001.0            |0.0               |-19998.0          |
|max    |2024.0            |97.0              |9999999.0         |
+-------+------------------+------------------+------------------+



Our dataset captures survey results between 2001 and 2024. Our age range is between 0 and 97 years old with an average age of approximately 41 years old. Our total income ranges from -19998 to 9999999 dollars. The data stored in the INCTOT column only has 7 digits appropriated. TODO: Check the website for INCTOT,

### Counting Nulls

In [9]:
all_null_counts = df.select([F.count(F.when(F.isnan(c) | F.col(c).isNull(), c)).alias(c) for c in df.columns])
all_null_counts.show(truncate=False)

+----+--------+---+---+----+----+------+
|YEAR|STATEFIP|SEX|AGE|RACE|EDUC|INCTOT|
+----+--------+---+---+----+----+------+
|0   |0       |0  |0  |0   |0   |0     |
+----+--------+---+---+----+----+------+



We can see that there are no null values in these columns. However, it remains to be seen if the unavailable data defined by the coding schemes (e.g. 99 for EDUC column) are present in the data. This will be known in the next section.

### Counting Categorical Columns

In [17]:
#While not exactly a categorical column, the year can still act as a category
df.groupBy("YEAR").count().sort("YEAR").show(df.count(), truncate=False)

+------+-------+
|YEAR  |count  |
+------+-------+
|2001.0|1192206|
|2002.0|1074628|
|2003.0|1194928|
|2004.0|1194354|
|2005.0|2878380|
|2006.0|2969741|
|2007.0|2994662|
|2008.0|3000657|
|2009.0|3030728|
|2010.0|3061692|
|2011.0|3112017|
|2012.0|3113030|
|2013.0|3132795|
|2014.0|3132610|
|2015.0|3147005|
|2016.0|3156487|
|2017.0|3190040|
|2018.0|3214539|
|2019.0|3239553|
|2020.0|2641054|
|2021.0|3252599|
|2022.0|3373378|
|2023.0|3405809|
|2024.0|3422888|
+------+-------+



In [19]:
df.groupBy("STATEFIP").count().sort("STATEFIP").show(df.count(), truncate=False)

+--------+-------+
|STATEFIP|count  |
+--------+-------+
|1.0     |1024336|
|2.0     |169583 |
|4.0     |1401893|
|5.0     |625593 |
|6.0     |7764652|
|8.0     |1130393|
|9.0     |764755 |
|10.0    |217185 |
|11.0    |151967 |
|12.0    |4085070|
|13.0    |2048497|
|15.0    |326018 |
|16.0    |369648 |
|17.0    |2704011|
|18.0    |1421822|
|19.0    |718263 |
|20.0    |644038 |
|21.0    |984880 |
|22.0    |949018 |
|23.0    |298374 |
|24.0    |1259877|
|25.0    |1456779|
|26.0    |2154755|
|27.0    |1183607|
|28.0    |657251 |
|29.0    |1316781|
|30.0    |239617 |
|31.0    |433678 |
|32.0    |583505 |
|33.0    |309490 |
|34.0    |1880416|
|35.0    |418393 |
|36.0    |4128739|
|37.0    |2073335|
|38.0    |187329 |
|39.0    |2530318|
|40.0    |795856 |
|41.0    |845684 |
|42.0    |2752179|
|44.0    |248965 |
|45.0    |1017169|
|46.0    |229292 |
|47.0    |1390874|
|48.0    |5304688|
|49.0    |646077 |
|50.0    |164591 |
|51.0    |1747241|
|53.0    |1521890|
|54.0    |420926 |
|55.0    |12

In [20]:
df.groupBy("SEX").count().sort("SEX").show(truncate=False)

+---+--------+
|SEX|count   |
+---+--------+
|1.0|32753559|
|2.0|34372221|
+---+--------+



In [21]:
df.groupBy("RACE").count().sort("RACE").show(truncate=False)

+----+--------+
|RACE|count   |
+----+--------+
|1.0 |50377188|
|2.0 |6582883 |
|3.0 |719191  |
|4.0 |830516  |
|5.0 |181559  |
|6.0 |2437481 |
|7.0 |2900008 |
|8.0 |2834320 |
|9.0 |262634  |
+----+--------+



In [22]:
df.groupBy("EDUC").count().sort("EDUC").show(truncate=False)

+----+--------+
|EDUC|count   |
+----+--------+
|0.0 |4282670 |
|1.0 |5186304 |
|2.0 |4927072 |
|3.0 |1611678 |
|4.0 |1805252 |
|5.0 |1934711 |
|6.0 |19994469|
|7.0 |7857403 |
|8.0 |4113044 |
|10.0|9542074 |
|11.0|5871103 |
+----+--------+



All categorical variables do not have any code scheme defined missing variables. All data of interest are present.